# 01 — Data Exploration and Data Extraction
**Project:** Transactional Fraud Detection Analysis  
**Author:** Data Analyst & Machine Learning Engineering Intern  
**Objective:** Ingest the European Credit Card Fraud dataset, discover its schema, perform data quality validation, and store the structured data into SQLite for relational SQL analysis.



In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3

# Set root directory
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import DataLoader
from src.utils import get_logger

logger = get_logger("ExplorationNotebook")
print("Environment initialized successfully.")



## 1. Load Raw Dataset
We utilize the modular `DataLoader` class to load the credit card transactions. The dataset contains 284,807 transactions with 30 anonymized numerical features and a binary target `Class` (0 = Legitimate, 1 = Fraudulent).



In [ ]:
loader = DataLoader()
df_raw = loader.load_data()

print(f"Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()



## 2. Schema and Data Quality Profiling
Let's inspect data types, memory consumption, missing values, duplicates, and statistical summaries.



In [ ]:
summary = loader.inspect_data(df_raw)

print(f"Memory Usage: {summary['memory_usage_mb']} MB")
print(f"Total Missing Values: {summary['total_missing_values']}")
print(f"Total Duplicate Rows: {summary['duplicate_rows']:,}")
print(f"Target Breakdown: {summary['target_summary']['counts']}")
print(f"Fraud Rate: {summary['target_summary']['fraud_rate_pct']:.4f}%")



## 3. Relational Storage (SQLite) & SQL Extraction
Cleaned and structured records are persisted into a local SQLite database (`fraud_detection.db`) enabling fast SQL-based analytical queries.



In [ ]:
loader.save_to_sqlite(df_raw, table_name="transactions")

# Demonstrate SQL query extraction
query = """
SELECT 
    COUNT(*) AS total_tx,
    SUM(is_fraud) AS fraud_count,
    ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 4) AS fraud_rate_pct,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(MAX(amount), 2) AS max_amount
FROM transactions;
"""
res_df = loader.run_sql_query(query)
res_df



## 4. Key Takeaways & Analytical Summary
1. **Extreme Imbalance:** Fraudulent transactions represent only ~0.17% of total volume (1 fraud per ~577 legitimate transactions). Accuracy is entirely inappropriate as an evaluation metric.
2. **Data Integrity:** No null values present in raw format; 1,081 duplicate transactions identified for cleaning in Phase 2.
3. **Storage:** Successfully integrated SQLite database with indexed fraud and amount columns.

